# 2. Propose and review individual-plant instances
Existing semantic masks do not distinguish adjacent plants. SAM proposals and connected components below are **drafts**, never ground truth. Dense weeds may require substantial manual annotation. Review complete images, including plants missed by proposals. Group disconnected visible leaves of the same plant; separate touching plants when identity is defensible. Do not invent occluded regions. Mark uncertain regions and resolve them before approving an image.

Proposals use a pretrained model only. Validation and test annotations must be reviewed before fine-tuning and remain fixed afterward. The raw semantic mask is displayed as a guide, not imposed on final reviewed boundaries. All annotation and proposal writes are explicitly enabled.


In [ ]:
from pathlib import Path
import json
import numpy as np
from PIL import Image
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'dataset_notebooks').is_dir() and (p / 'notebooks/reproduction.py').is_file())
sys.path.insert(0, str(ROOT / 'dataset_notebooks'))
from runtime import fresh_run, write_json, read_json, sha256, environment, load_model, SAM3_REVISION


In [ ]:
AUDIT_PATH = ROOT/'experiments/reproductions/sam3-audit-v1/audit.json'
DATASET_ROOT = ROOT/'data/tobacco_original'
RUN_NAME = 'sam3-annotations-v1'
ANNOTATIONS = ROOT/'experiments/reproductions'/RUN_NAME
CREATE_WORKSPACE = False
RUN_PROPOSALS = False
OPEN_REVIEW = False
RUN_EXPORT = False
ALLOW_WEIGHT_DOWNLOAD = False
PRETRAINED_CHECKPOINT = None  # Local sam3.pt, or explicitly allow authenticated HF download.
PROPOSAL_KEYS = []  # Explicit image keys; empty means all images.
REVIEW_KEY = 'c01_0001'
REVIEWER = ''
EXPORT_RUN_NAME = 'sam3-reviewed-export-v1'


## Draft proposals

Connected components provide an editable CPU starting point. GPU proposals use pretrained SAM 3 and the semantic guide. Neither route produces approved instances.


In [ ]:
def draft_components(semantic):
    import cv2
    labels=np.zeros(semantic.shape,np.int32); classes={}; next_id=1
    for category in (1,2):
        count, components=cv2.connectedComponents((semantic==category).astype('uint8'),connectivity=8)
        for k in range(1,count):
            labels[components==k]=next_id; classes[next_id]=category; next_id+=1
    return labels,classes


def draft_sam(model, image, semantic):
    import torch
    from sam3.model.sam3_image_processor import Sam3Processor
    processor=Sam3Processor(model,confidence_threshold=0.15)
    proposals=[]
    with torch.inference_mode(), torch.autocast('cuda',dtype=torch.bfloat16):
        state=processor.set_image(image)
        for category,prompt in [(1,'tobacco plant'),(2,'weed plant')]:
            processor.reset_all_prompts(state)
            output=processor.set_text_prompt(prompt=prompt,state=state)
            for mask,score in zip(output['masks'],output['scores']):
                mask=mask.squeeze(0).detach().cpu().numpy().astype(bool)
                # Semantic guide filters gross class errors; reviewer can restore pixels.
                mask &= semantic==category
                if mask.any(): proposals.append((float(score),category,mask))
    labels=np.zeros(semantic.shape,np.int32); classes={}
    for score,category,mask in sorted(proposals,key=lambda x:-x[0]):
        mask &= labels==0
        if mask.any():
            k=len(classes)+1;labels[mask]=k;classes[k]=category
    return labels,classes


## Review records

Each save creates a new revision with an image hash and annotation-array hash. Approval requires a reviewer, notes, valid class IDs, and no unresolved pixels.


In [ ]:
def validate_instances(labels, classes, unresolved=None):
    if labels.ndim!=2 or not np.issubdtype(labels.dtype,np.integer) or labels.min()<0:
        raise ValueError('Instance map must be a nonnegative 2D integer array.')
    present=set(map(int,np.unique(labels)))-{0}
    if present!=set(map(int,classes)) or any(int(c) not in (1,2) for c in classes.values()):
        raise ValueError('Instance IDs and class table disagree.')
    if unresolved is not None and (unresolved.shape!=labels.shape or unresolved.any()):
        raise ValueError('Resolve uncertain pixels before approval.')


def save_review(folder,row,labels,classes,unresolved,reviewer,approved,notes):
    from datetime import datetime, timezone
    folder=Path(folder)/'reviews'/row['key'];folder.mkdir(parents=True,exist_ok=True)
    classes={int(k):int(v) for k,v in classes.items() if np.any(labels==int(k))}
    validate_instances(labels,classes,unresolved if approved else None)
    if approved and (not reviewer.strip() or not notes.strip()):
        raise ValueError('Approval requires reviewer name and review notes.')
    version=max([int(p.stem.split('-')[1]) for p in folder.glob('revision-*.json')]+[0])+1
    base=f'revision-{version:04d}'
    array_path=folder/(base+'.npz')
    with array_path.open('xb') as stream:
        np.savez_compressed(stream,instances=labels.astype(np.int32),unresolved=unresolved.astype(bool))
    meta={'key':row['key'],'image_sha256':row['image_sha256'],'classes':classes,
        'reviewer':reviewer,'approved':bool(approved),'notes':notes,
        'updated_utc':datetime.now(timezone.utc).isoformat(),'array_file':array_path.name,
        'array_sha256':sha256(array_path),'source':'human_review' if approved else 'draft'}
    write_json(folder/(base+'.json'),meta)
    return meta


def load_review(folder,row):
    folder=Path(folder)/'reviews'/row['key'];paths=sorted(folder.glob('revision-*.json'))
    if not paths: return None
    meta=read_json(paths[-1]);p=folder/meta['array_file']
    if meta['image_sha256']!=row['image_sha256'] or sha256(p)!=meta['array_sha256']:
        raise ValueError('Annotation/image content changed.')
    with np.load(p,allow_pickle=False) as z:
        labels=z['instances'].copy();unresolved=z['unresolved'].copy()
    return labels,{int(k):v for k,v in meta['classes'].items()},unresolved,meta


## Portable training artifacts

Export only approved complete images. Validation and test coverage must be complete, and exact duplicate images are excluded. COCO RLE retains disconnected parts and holes.


In [ ]:
def export_reviewed(audit, data_root, annotations, output):
    import shutil
    from pycocotools import mask as mask_api
    if not audit['split_review']['approved'] or audit['cross_split_duplicates']:
        raise ValueError('Dataset split review is not approved.')
    approved=[];missing_eval=[]
    for row in audit['rows']:
        if row.get('duplicate_of'):continue
        review=load_review(annotations,row)
        if review is None or not review[3]['approved']:
            if row['split']!='train':missing_eval.append(row['key'])
            continue
        labels,classes,uncertain,meta=review
        validate_instances(labels,classes,uncertain)
        if labels.shape!=(row['height'],row['width']):raise ValueError('Annotation dimensions changed.')
        if sha256(Path(data_root)/row['image'])!=row['image_sha256']:raise ValueError('Input changed.')
        approved.append((row,labels,classes,meta))
    if missing_eval:raise ValueError(f'Entire validation/test sets must be reviewed: {len(missing_eval)} missing.')
    if not any(r['split']=='train' for r,*_ in approved):raise ValueError('No reviewed training images.')
    output=Path(output);output.mkdir(parents=True,exist_ok=False)
    for split in ('train','val','test'):
        folder=output/split;folder.mkdir()
        coco={'info':{'description':'Human-reviewed tobacco/weed instances'},'licenses':[],
            'images':[],'annotations':[], 'categories':[{'id':1,'name':'tobacco plant'},{'id':2,'name':'weed plant'}]}
        for row,labels,classes,meta in approved:
            if row['split']!=split:continue
            filename=row['key']+'.png';shutil.copy2(Path(data_root)/row['image'],folder/filename)
            coco['images'].append({'id':row['id'],'file_name':filename,'width':row['width'],
                'height':row['height'],'campaign':row['campaign'],'source_key':row['key'],
                'image_sha256':row['image_sha256'],'annotation_provenance':meta})
            for instance,category in classes.items():
                rle=mask_api.encode(np.asfortranarray((labels==instance).astype('uint8')))
                area=int(mask_api.area(rle));bbox=mask_api.toBbox(rle).tolist();rle['counts']=rle['counts'].decode('ascii')
                coco['annotations'].append({'id':len(coco['annotations'])+1,'image_id':row['id'],
                    'category_id':category,'segmentation':rle,'bbox':bbox,'area':area,'iscrowd':0})
        write_json(folder/'annotations.json',coco)
    write_json(output/'provenance.json',{'sam3_revision':SAM3_REVISION,
        'split_review':audit['split_review'],'approved_keys':[r['key'] for r,*_ in approved],
        'excluded_duplicates':{r['key']:r['duplicate_of'] for r in audit['rows'] if r.get('duplicate_of')},
        'excluded_unreviewed_train':[r['key'] for r in audit['rows'] if r['split']=='train' and r['key'] not in {a[0]['key'] for a in approved}],
        'audit_rows':audit['rows']})
    return output


## Interactive editor

Paint visible plant pixels at source resolution. New IDs split plants; merge reunites disconnected parts. Mark all inspected only after checking the complete image; approval is a separate action.


In [ ]:
def review_widget(image,semantic,labels,classes,unresolved,on_save):
    import cv2
    import ipywidgets as w
    from ipycanvas import Canvas
    from IPython.display import display
    labels=labels.copy();classes=dict(classes);unresolved=unresolved.copy()
    rgb=np.array(image.convert('RGB'));height,width=labels.shape
    canvas=Canvas(width=width,height=height);guide=Canvas(width=width,height=height)
    active=w.IntText(value=max(classes,default=0)+1,description='Instance')
    category=w.Dropdown(options=[('Tobacco',1),('Weed',2)],description='Class')
    mode=w.Dropdown(options=['paint','erase','select','uncertain','resolve'],description='Tool')
    radius=w.IntSlider(value=4,min=1,max=30,description='Brush')
    merge=w.Text(description='Merge IDs',placeholder='e.g. 2, 5; into active ID')
    notes=w.Textarea(description='Notes');status=w.Output()
    new=w.Button(description='New / split');merge_button=w.Button(description='Merge')
    reclass=w.Button(description='Set class');save=w.Button(description='Save draft');approve=w.Button(description='Approve complete image')
    undo=w.Button(description='Undo');inspected=w.Button(description='Mark all inspected');delete=w.Button(description='Delete active');history=[];pressed=[False]
    def snapshot():
        history.append((labels.copy(),dict(classes),unresolved.copy()))
        if len(history)>30:history.pop(0)
    def redraw():
        out=rgb.astype(float).copy()
        for k in np.unique(labels):
            if k==0:continue
            color=np.random.default_rng(int(k)).integers(40,255,3)
            out[labels==k]=.55*out[labels==k]+.45*color
        out[unresolved]=(255,0,255)
        canvas.put_image_data(out.astype('uint8'))
        for k in np.unique(labels):
            if not k:continue
            yy,xx=np.where(labels==k);canvas.fill_style='white';canvas.font='12px sans-serif'
            canvas.fill_text(f'{k}:{classes.get(int(k),"?")}',float(xx.mean()),float(yy.mean()))
        guide_rgb=rgb.copy();guide_rgb[semantic==1]=(0,255,255);guide_rgb[semantic==2]=(255,255,0)
        guide.put_image_data((.6*rgb+.4*guide_rgb).astype('uint8'))
    def draw(x,y):
        x,y=int(x),int(y)
        if not(0<=x<width and 0<=y<height):return
        if mode.value=='select':
            active.value=int(labels[y,x]);category.value=classes.get(active.value,1);return
        brush=np.zeros(labels.shape,np.uint8);cv2.circle(brush,(x,y),radius.value,1,-1);mask=brush.astype(bool)
        if mode.value=='paint':
            if active.value<=0:return
            classes[active.value]=category.value;labels[mask]=active.value
        elif mode.value=='erase':labels[mask]=0
        elif mode.value=='uncertain':unresolved[mask]=True
        elif mode.value=='resolve':unresolved[mask]=False
        redraw()
    def down(x,y):snapshot();pressed[0]=True;draw(x,y)
    def move(x,y):
        if pressed[0]:draw(x,y)
    def up(x,y):pressed[0]=False
    canvas.on_mouse_down(down);canvas.on_mouse_move(move);canvas.on_mouse_up(up)
    def add(_):active.value=max(classes,default=0)+1
    def join(_):
        with status:
            try:
                ids=[int(x.strip()) for x in merge.value.split(',')]
                if active.value<=0 or not ids or any(k not in classes for k in ids):raise ValueError('Select existing IDs and a positive target.')
                snapshot();labels[np.isin(labels,ids)]=active.value;classes[active.value]=category.value;redraw()
            except Exception as e:print(e)
    def change(_):
        snapshot();classes[active.value]=category.value;redraw()
    def back(_):
        if history:
            old,c,u=history.pop();labels[:]=old;classes.clear();classes.update(c);unresolved[:]=u;redraw()
    def persist(approved):
        with status:
            try:print(on_save(labels,classes,unresolved,approved,notes.value)['updated_utc'])
            except Exception as e:print(type(e).__name__,str(e))
    def clear_uncertainty(_):snapshot();unresolved[:]=False;redraw()
    def remove_instance(_):snapshot();labels[labels==active.value]=0;redraw()
    inspected.on_click(clear_uncertainty);delete.on_click(remove_instance)
    new.on_click(add);merge_button.on_click(join);reclass.on_click(change);undo.on_click(back)
    save.on_click(lambda _:persist(False));approve.on_click(lambda _:persist(True))
    redraw()
    display(w.VBox([w.HTML('Left: editable instances; right: original semantic guide. New / split creates a new ID; paint over the part to split. Magenta = unresolved.'),
        w.HBox([canvas,guide]),w.HBox([active,category,mode]),w.HBox([radius,new,reclass,delete,undo]),
        w.HBox([merge,merge_button]),notes,w.HBox([inspected,save,approve]),status]))
    return canvas


In [ ]:
if CREATE_WORKSPACE:
    audit=read_json(AUDIT_PATH)
    if not audit['split_review']['approved']:raise ValueError('Approve dataset split in notebook 1.')
    fresh_run(ROOT,RUN_NAME)
    write_json(ANNOTATIONS/'workspace.json',{'audit_sha256':sha256(AUDIT_PATH),'sam3_revision':SAM3_REVISION})
if RUN_PROPOSALS:
    if not (ANNOTATIONS/'workspace.json').is_file():raise FileNotFoundError('Create annotation workspace first.')
    audit=read_json(AUDIT_PATH)
    model=load_model(PRETRAINED_CHECKPOINT,ALLOW_WEIGHT_DOWNLOAD)
    for row in audit['rows']:
        if row.get('duplicate_of'):continue
        if PROPOSAL_KEYS and row['key'] not in PROPOSAL_KEYS:continue
        if load_review(ANNOTATIONS,row) is not None:continue  # Never overwrite reviews.
        image=Image.open(DATASET_ROOT/row['image']).convert('RGB')
        semantic=np.array(Image.open(DATASET_ROOT/row['semantic_mask']))
        labels,classes=draft_sam(model,image,semantic)
        save_review(ANNOTATIONS,row,labels,classes,semantic>0,'',False,'Unreviewed SAM 3 proposal; resolve uncertain foreground.')
    del model
if OPEN_REVIEW:
    audit=read_json(AUDIT_PATH)
    if not (ANNOTATIONS/'workspace.json').is_file():raise FileNotFoundError('Create annotation workspace first.')
    if sha256(AUDIT_PATH)!=read_json(ANNOTATIONS/'workspace.json')['audit_sha256']:raise ValueError('Audit changed since workspace creation.')
    row=next(r for r in audit['rows'] if r['key']==REVIEW_KEY)
    image=Image.open(DATASET_ROOT/row['image']).convert('RGB')
    semantic=np.array(Image.open(DATASET_ROOT/row['semantic_mask']))
    existing=load_review(ANNOTATIONS,row)
    if existing: labels,classes,unresolved,_=existing
    else: labels,classes=draft_components(semantic);unresolved=semantic>0
    editor=review_widget(image,semantic,labels,classes,unresolved,
        lambda labels,classes,uncertain,approved,notes:save_review(ANNOTATIONS,row,labels,classes,uncertain,REVIEWER,approved,notes))
if RUN_EXPORT:
    audit=read_json(AUDIT_PATH)
    if sha256(AUDIT_PATH)!=read_json(ANNOTATIONS/'workspace.json')['audit_sha256']:raise ValueError('Audit changed since workspace creation.')
    export_path=ROOT/'experiments/reproductions'/EXPORT_RUN_NAME
    print(export_reviewed(audit,DATASET_ROOT,ANNOTATIONS,export_path))
if not any([CREATE_WORKSPACE,RUN_PROPOSALS,OPEN_REVIEW,RUN_EXPORT]):print('Annotation workflow disabled.')


## Export contract
The training notebook consumes `train/annotations.json` and `val/annotations.json`; final evaluation additionally consumes `test/annotations.json`. Each split contains copied PNG images with unique campaign-qualified filenames. COCO RLE preserves holes and disconnected visible parts. The export excludes unreviewed training images and refuses incomplete validation/test annotations. Every approval saves a new revision; use **Save draft** to revoke approval after edits, then approve again after review. Model-produced proposals alone cannot pass the approval gate.
